### New algorithm for enumerating every $\Z_2^n$-colorable PL~spheres of Picard number $5$

In [1]:
using Oscar
using ProgressBars
using SparseArrays
using Serialization


  ___   ___   ___    _    ____
 / _ \ / __\ / __\  / \  |  _ \  | Combining and extending ANTIC, GAP,
| |_| |\__ \| |__  / ^ \ |  ´ /  | Polymake and Singular
 \___/ \___/ \___//_/ \_\|_|\_\  | Type "?Oscar" for more information
o--------o-----o-----o--------o  | Documentation: https://docs.oscar-system.org
  S Y M B O L I C   T O O L S    | Version 1.4.1


In [2]:
function boundary_incidence_facets_to_ridges(facets::Vector{Vector{Int}})
    d = length(facets[1]) - 1  # facet dimension
    @assert all(length(f) == d+1 for f in facets) "Need a pure complex (all facets same size)."

    # collect ridges (each facet contributes its (d-1)-subfaces by deleting one vertex)
    ridge_dict = Dict{Vector{Int}, Int}()  # ridge -> row index
    ridges = Vector{Vector{Int}}()
    for f in facets
        for k in eachindex(f)
            r = [f[i] for i in eachindex(f) if i != k]  # already sorted since f is sorted
            if !haskey(ridge_dict, r)
                push!(ridges, r)
                ridge_dict[r] = length(ridges)
            end
        end
    end

    m = length(ridges)
    n = length(facets)

    # build sparse 0/1 matrix (m x n): ridges x facets
    I = Int[]   # row indices
    J = Int[]   # col indices
    V = Int8[]  # values (all ones)

    for (j, f) in pairs(facets)
        for k in eachindex(f)
            r = [f[i] for i in eachindex(f) if i != k]
            i = ridge_dict[r]
            push!(I, i)
            push!(J, j)
            push!(V, true)
        end
    end

    A = sparse(I, J, V, m, n)  # SparseMatrixCSC{Bool} 

    return ridges, A
end



boundary_incidence_facets_to_ridges (generic function with 1 method)

In [3]:
function kernel_basis_mod2_sparse(A)
    m, n = size(A)

    # Store rows as sparse BitVectors (using sets of column indices)
    rows = [Set{Int}() for _ in 1:m]
    
    # Build rows from A mod 2
    @inbounds for j in 1:n
        for p in nzrange(A, j)
            i = rowvals(A)[p]
            if isodd(A.nzval[p])
                if j in rows[i]
                    delete!(rows[i], j)  # XOR: 1⊕1=0
                else
                    push!(rows[i], j)     # XOR: 0⊕1=1
                end
            end
        end
    end

    # RREF over GF(2), recording pivot columns and pivot rows
    pivcol = Int[]
    pivrow = Int[]
    row = 1
    
    @inbounds for col in 1:n
        row > m && break

        # Find pivot in this column at/under current row
        piv = 0
        for r in row:m
            if col in rows[r]
                piv = r
                break
            end
        end
        piv == 0 && continue

        # Swap rows
        if piv != row
            rows[row], rows[piv] = rows[piv], rows[row]
        end

        push!(pivcol, col)
        push!(pivrow, row)

        # Eliminate this column in all other rows (RREF)
        pivot_row = rows[row]
        for r in 1:m
            if r != row && col in rows[r]
                # XOR rows[r] with pivot_row
                symdiff!(rows[r], pivot_row)
            end
        end

        row += 1
    end

    # Free columns
    pivot_set = Set(pivcol)
    free_cols = [j for j in 1:n if !(j in pivot_set)]

    basis = BitVector[]
    isempty(free_cols) && return basis

    # Build one kernel vector per free column
    @inbounds for f in free_cols
        x_set = Set{Int}([f])  # Sparse representation of x

        # Back-substitute using RREF rows
        for t in length(pivcol):-1:1
            c = pivcol[t]
            r = pivrow[t]
            
            # Compute parity: count elements in rows[r] ∩ x_set, excluding c
            row_r = rows[r]
            parity = false
            for j in row_r
                if j != c && j in x_set
                    parity = !parity
                end
            end
            
            # Set x[c] based on parity
            if parity
                push!(x_set, c)
            else
                delete!(x_set, c)
            end
        end

        # Convert sparse set to BitVector
        x = falses(n)
        for j in x_set
            x[j] = true
        end
        push!(basis, x)
    end

    return basis
end

kernel_basis_mod2_sparse (generic function with 1 method)

In [4]:
using SparseArrays

"""
    kernel_elements_with_Ax_in_0_or_2_noprune(A, basis) -> Vector{BitVector}

Full enumeration WITHOUT pruning.

- A is strictly 0/1 and used over ℤ for Ax.
- basis is a GF(2)-basis of ker(A mod 2), given as Vector{BitVector}.
- Enumerates all nonzero x in the GF(2)-span of `basis` and returns those with Ax ∈ {0,2}^m.
- Complexity: Θ(2^k * (cost to update/check)), where k = length(basis).
"""
function kernel_elements_with_Ax_in_0_or_2(
    A::SparseMatrixCSC{<:Integer},
    basis::Vector{BitVector},
)
    m, n = size(A)
    k = length(basis)
    # println("Dimension of the kernel: ",k)
    @assert all(length(b) == n for b in basis) "Basis vectors must have length = #cols(A)."

    # column -> incident rows (A is 0/1)
    rv = rowvals(A)
    colrows = Vector{Vector{Int}}(undef, n)
    for j in 1:n
        colrows[j] = [rv[p] for p in nzrange(A, j)]
    end

    # supports of basis vectors (columns they toggle)
    supp = Vector{Vector{Int}}(undef, k)
    for t in 1:k
        supp[t] = findall(basis[t])
    end

    x = falses(n)
    counts = zeros(Int16, m)  # Ax over ℤ
    sols = BitVector[]

    function toggle_basis!(t::Int)
        @inbounds for j in supp[t]
            turning_on = !x[j]
            x[j] = turning_on
            δ = turning_on ? Int16(1) : Int16(-1)
            for i in colrows[j]
                counts[i] += δ
            end
        end
    end

    @inline function feasible_final()::Bool
        @inbounds for i in 1:m
            c = counts[i]
            if !(c == 0 || c == 2)
                return false
            end
        end
        return true
    end

    function dfs(t::Int, any_one::Bool)
        if t > k
            if any_one && feasible_final()
                push!(sols, copy(x))
            end
            return
        end

        # branch 0: do not include basis[t]
        dfs(t + 1, any_one)

        # branch 1: include basis[t]
        toggle_basis!(t)
        dfs(t + 1, true)
        toggle_basis!(t)  # toggle back (self-inverse over GF(2))
    end

    dfs(1, false)  # excludes zero vector by any_one=false
    return sols
end


kernel_elements_with_Ax_in_0_or_2

In [14]:
# If homology is UNREDUCED in your setup (most common):
function is_homology_sphere(K::Oscar.SimplicialComplex)
    b = betti_numbers(K)
    d = dim(K)
    for i in 0:d
        if i == 0 || i == d
            b[i+1] == 1 || return false
        else
            b[i+1] == 0 || return false
        end
    end
    return true
end


is_homology_sphere (generic function with 1 method)

In [9]:
using Polymake
const topaz = Polymake.topaz
using ProgressMeter
using Nemo
F = GF(2)


# Build canonical vertex list present in a matroid (sorted)
function vertex_list_of_bases(bases::Vector{Vector{Int}})
    s = Set{Int}()
    for b in bases
        for v in b
            push!(s, v)
        end
    end
    return sort(collect(s))
end

# Construct a matrix from a binary matroid

function binary_matrix_representation(M; B=nothing)
    is_binary(M) || error("Matroid is not binary")

    # ground set
    E = collect(matroid_groundset(M))
    n = length(E)

    # choose basis
    if B === nothing
        B = first(bases(M))
    else
        length(B) == rank(M) || error("Provided set is not a basis")
    end

    r = length(B)

    # index maps
    row_of = Dict(b => i for (i,b) in enumerate(B))
    col_of = Dict(e => j for (j,e) in enumerate(E))

    # initialize matrix
    A = matrix(F,zeros(Int, (r, n)))

    # basis columns = identity
    for b in B
        A[row_of[b], col_of[b]] = F(1)
    end

    # non-basis columns via fundamental circuits
    for e in E
        e in B && continue
        C = fundamental_circuit(M, B, e)
        for b in C
            b in B || continue
            A[row_of[b], col_of[e]] = F(1)
        end
    end

    return A, B, E
end

# ---------------------------
# using Topaz for isom
# ---------------------------

function bases_to_topaz_complex(bases::Vector{Vector{Int}})
    # collect and sort vertices
    verts = sort!(unique(vcat(bases...)))

    # map vertex labels to 0-based indices
    vmap = Dict(v => i-1 for (i,v) in enumerate(verts))

    facets = [ [vmap[x] for x in B] for B in bases ]
    return topaz.SimplicialComplex(FACETS = facets), verts
end

function topaz_isomorphism(basesA, basesB)
    SC_A, vertsA = bases_to_topaz_complex(basesA)
    SC_B, vertsB = bases_to_topaz_complex(basesB)
    length(vertsA) == length(vertsB) || return nothing
    T = topaz.find_facet_vertex_permutations(SC_A, SC_B)
    if T===nothing
        return nothing
    end
    _,p=T
    perm = Dict(
        vertsA[i] => vertsB[p[i]+1]
        for i in eachindex(vertsA)
    )
    return perm
end






# ---------------------------
# Main builder
# ---------------------------
"""
build_iso_db!(Iso_DB, mat_DB; ms = sort(collect(keys(mat_DB))), verbose=false)

- Iso_DB will be mutated (create empty Dict before passing).
- mat_DB: Dict{Int, Vector{Vector{Vector{Int}}}} mapping m -> list of matroid-bases
- For each m in ms (skips if m-1 not present), and each k in mat_DB[m],
  finds first vertex v in 1:m such that corank(contract_at_vertex(M,v)) >= corank(M),
  computes contraction, then finds l and a permutation mapping contraction -> some mat_DB[m-1][l].
- Stores Iso_DB[m][k] = (l, mapping::Dict{Int,Int}) or (-1, nothing) if not found.
"""
function build_iso_db!(Iso_DB::Dict{Int,Dict{Int,Tuple{Int,Int,Any}}}, mat_DB::Dict{Int,Vector{Vector{Vector{Int}}}}; ms=nothing, verbose=false)
    # ms === nothing && (ms = sort(collect(keys(mat_DB))))
    for m in ms
        println(m)
        if !haskey(mat_DB, m-1)
            verbose && @info("Skipping m=$m because mat_DB[$(m-1)] not present")
            continue
        end
        Iso_DB[m] = Dict{Int,Tuple{Int,Any}}()
        @showprogress desc="Computing m=$(m)..." for (k, Mbases) in enumerate(mat_DB[m])
            V=vertex_list_of_bases(Mbases)
            M = matroid_from_bases(Mbases,V)
            found_v = nothing
            Mv_chosen = nothing
            # pick smallest v in 1:m satisfying corank(contract) >= corank(M)
            v_chosen=nothing
            for v in V
                if v in coloops(M)
                    continue
                end
                Mv = deletion(M,v)
                if rank(Mv)==rank(M)
                    found_v=true
                    v_chosen=v
                    Mv_chosen=Mv
                    break
                end
            end
            if found_v === nothing
                v_chosen = V[1]
            end
            Mv_chosen = deletion(M,v_chosen)
            deletion_bases = bases(Mv_chosen)

            # search through mat_DB[m-1] for an isomorphism
            perm = nothing
            found_index = -1
            found_isom = false
            for (l, target_bases) in enumerate(mat_DB[m-1])
                M2 = matroid_from_bases(target_bases,vertex_list_of_bases(target_bases))
                if !is_isomorphic(Mv_chosen,M2)
                    continue
                end
                perm = topaz_isomorphism(target_bases,deletion_bases)
                if perm !== nothing
                    found_index = l
                    break
                end
            end

            if perm === nothing
                Iso_DB[m][k] = (-1, v_chosen, nothing)
                A,_,_ = binary_matrix_representation(M)
                display(A)
                verbose && @warn(
                    "No isomorphic matroid found for contraction  m=$m, k=$k at v=$v_chosen"
                )
            else
                Iso_DB[m][k] = (found_index,v_chosen, perm)
                # verbose && @info(
                #     "Found iso: m=$m, k=$k -> m-1 index=$found_index, v=$v_chosen"
                # )
            end
        end
    end
    return Iso_DB
end


build_iso_db!

In [12]:
function kernel_basis_echelon_prioritize(B, S)
    """
    Convert kernel basis B to echelon form over GF(2), prioritizing columns in S.
    Returns (B_ech, pivots) where:
    - B_ech is the echelon basis
    - pivots[i] is the pivot position (leading 1) of B_ech[i]
    - Columns in S are processed first to maximize forced coefficients
    """
    isempty(B) && return (BitVector[], Int[])
    
    n = length(B[1])
    k = length(B)
    
    # Copy basis vectors
    B_ech = [copy(b) for b in B]
    pivots = Int[]
    
    # Build column order: S columns first, then others
    cols_in_S = [j for j in 1:n if S[j]]
    cols_not_in_S = [j for j in 1:n if !S[j]]
    col_order = vcat(cols_in_S, cols_not_in_S)
    
    current_row = 1
    for col in col_order
        current_row > k && break
        
        # Find a vector with 1 at position col (at or below current_row)
        piv = 0
        for r in current_row:k
            if B_ech[r][col]
                piv = r
                break
            end
        end
        
        piv == 0 && continue  # No pivot in this column
        
        # Swap to current position
        if piv != current_row
            B_ech[current_row], B_ech[piv] = B_ech[piv], B_ech[current_row]
        end
        
        push!(pivots, col)
        
        # Eliminate this column in all other rows
        for r in 1:k
            if r != current_row && B_ech[r][col]
                B_ech[r] .⊻= B_ech[current_row]
            end
        end
        
        current_row += 1
    end
    
    return (B_ech, pivots)
end

function enumerate_kernel_with_constraints(A, B, S)
    m, n = size(A)
    results = BitVector[]
    
    if isempty(B)
        x = falses(n)
        if all(.!S)
            Ax = compute_Ax_integer(A, x)
            if all(v -> v == 0 || v == 2, Ax)
                push!(results, x)
            end
        end
        return results
    end
    
    # Convert basis to echelon form, prioritizing S columns
    B_ech, pivots = kernel_basis_echelon_prioritize(B, S)
    k = length(B_ech)
    
    # Determine which coefficients are forced by constraint S
    # If S[pivot[i]] = 1, then λ[i] must be 1
    forced_lambda = falses(k)
    free_indices = Int[]
    
    for i in 1:k
        piv = pivots[i]
        if S[piv]
            # This coefficient must be 1
            forced_lambda[i] = true
        else
            # This coefficient is free
            push!(free_indices, i)
        end
    end
    
    # Compute the base vector from forced coefficients
    x_forced = falses(n)
    for i in 1:k
        if forced_lambda[i]
            x_forced .⊻= B_ech[i]
        end
    end
    
    # Check if forced coefficients already satisfy all S constraints
    for j in 1:n
        if S[j] && !x_forced[j]
            # j ∈ S but x_forced[j] = 0
            # This means j is not a pivot, so we need free variables to set it
            # But if we prioritized S columns, this shouldn't happen unless
            # the constraint is unsatisfiable
            can_fix = false
            for idx in free_indices
                if B_ech[idx][j]
                    can_fix = true
                    break
                end
            end
            if !can_fix
                return results  # Impossible to satisfy S
            end
        end
    end
    
    # Now enumerate only over free coefficients
    num_free = length(free_indices)
    
    for mask in 0:(2^num_free - 1)
        x = copy(x_forced)
        
        # Add contribution from free variables
        for (bit_pos, idx) in enumerate(free_indices)
            if (mask >> (bit_pos - 1)) & 1 == 1
                x .⊻= B_ech[idx]
            end
        end
        
        # Check constraint S
        satisfies_S = true
        for i in 1:n
            if S[i] && !x[i]
                satisfies_S = false
                break
            end
        end
        
        if !satisfies_S
            continue
        end
        
        # Check 0/2 constraint
        Ax = compute_Ax_integer(A, x)
        
        if all(v -> v == 0 || v == 2, Ax)
            push!(results, copy(x))
        end
    end
    
    return results
end

function compute_Ax_integer(A, x)
    m, n = size(A)
    result = zeros(Int, m)
    
    for j in 1:n
        if x[j]
            for p in nzrange(A, j)
                i = rowvals(A)[p]
                result[i] += A.nzval[p]
            end
        end
    end
    
    return result
end

compute_Ax_integer (generic function with 1 method)

In [10]:
global mat_DB = open("rank_4_simple_bin_mat_DB.jls", "r") do io
    deserialize(io)
end

iso_DB = Dict{Int,Dict{Int,Tuple{Int,Int,Any}}}()

build_iso_db!(iso_DB,mat_DB,ms=7:15,verbose=true)

open("rank_4_iso_DB_m_7-15.jls", "w") do io
    serialize(io, iso_DB)
end

7


Computing m=7... 100%|███████████████████████████████████| Time: 0:00:00


8
9
10


Computing m=10... 100%|██████████████████████████████████| Time: 0:00:00


11


Computing m=11... 100%|██████████████████████████████████| Time: 0:00:00


12
13
14
15


In [32]:
using Profile


global iso_DB = open("rank_4_iso_DB_m_7-15.jls", "r") do io
    deserialize(io)
end

function subset_bitvector(superset::Vector{Vector{Int}}, subset::Vector{Vector{Int}})
    n = length(superset)
    result = falses(n)
    
    j = 1  # index dans subset
    for i in 1:n
        if j <= length(subset) && superset[i] == subset[j]
            result[i] = true
            j += 1
        end
    end
    
    return result
end

function initialize_DB!(pseudo_manifolds_DB::Dict{Int,Vector{Vector{Bool}}},mat_DB::Dict{Int,Vector{Vector{Vector{Int}}}})
    mmin = minimum(collect(keys(mat_DB)))
    pseudo_manifolds_DB[mmin] = [enumerate_polygons(bases) for bases in mat_DB[mmin]]
    return pseudo_manifolds_DB
end


function build_finalDB_single_v!(pseudo_manifolds_DB::Dict{Int,Vector{Vector{BitVector}}},mat_DB::Dict{Int,Vector{Vector{Vector{Int}}}},iso_DB::Dict{Int,Dict{Int,Tuple{Int,Int,Any}}},mmax)
    mmin = minimum(collect(keys(mat_DB)))
    for m=mmin:mmax
        println(m)
        pseudo_manifolds_DB[m] = Vector{Vector{BitVector}}()
        @showprogress for (l,bases) in enumerate(mat_DB[m])
            # display(bases)
            V = vertex_list_of_bases(bases)
            compl_bases = [setdiff(V,base) for base in bases] # we need to complement to get the correct boundary matrix
            ridges, A = boundary_incidence_facets_to_ridges(compl_bases)  
            basis = kernel_basis_mod2_sparse(A)
            push!(pseudo_manifolds_DB[m], Vector{BitVector}())
            if m==mmin
                mandatory_facets_bit=falses(length(bases))
                all_solutions_bit = kernel_elements_with_Ax_in_0_or_2(A,basis)
                for K_bit in all_solutions_bit
                    K = simplicial_complex(collect(compl_bases[findall(K_bit)]))
                    if is_sphere(K)
                        push!(pseudo_manifolds_DB[m][l],K_bit)
                    end
                end
            else
                index_contraction, v_contract, perm = iso_DB[m][l]
                for L in pseudo_manifolds_DB[m-1][index_contraction]
                    # println(L)
                    mandatory_facets = [sort(vcat([perm[i] for i in facet_L],v_contract)) for facet_L in mat_DB[m-1][index_contraction][findall(L)]]
                    # print(mandatory_facets)
                    mandatory_facets_bit = subset_bitvector(bases, mandatory_facets)
                    all_solutions_bit = enumerate_kernel_with_constraints(A,basis,mandatory_facets_bit)
                    for K_bit in all_solutions_bit
                        K = simplicial_complex(collect(compl_bases[findall(K_bit)]))
                        if is_homology_sphere(K)
                            test = true
                            for v in vertex_list_of_bases([collect(facet) for facet in facets(K)])
                                if !is_homology_sphere(link_subcomplex(K,[v]))
                                    test=false
                                    break
                                end
                            end
                            if test
                                push!(pseudo_manifolds_DB[m][l],K_bit)
                            end
                        end
                    end
                end
            end
        end
    end

end

global pseudo_manifolds_DB = Dict{Int,Vector{Vector{BitVector}}}()

@profile build_finalDB_single_v!(pseudo_manifolds_DB,mat_DB,iso_DB,7)

using ProfileView
ProfileView.view()




6


Progress: 100%|█████████████████████████████████████████| Time: 0:00:00


7


Progress: 100%|█████████████████████████████████████████| Time: 0:01:48


LoadError: ArgumentError: Package ProfileView not found in current path.
- Run `import Pkg; Pkg.add("ProfileView")` to install the ProfileView package.